In [ ]:
import kagglehub

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

q3_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(q3_path)

In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
print(df.isnull().sum())
for col in df.columns[df.isnull().any()]:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
# Task 2: Write your code here:
print(f"Duplicates: {df.duplicated().sum()}")
df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
feature_cols = [col for col in df.columns if col != 'Target']
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

In [ ]:
# Task 5: Write your code here:
print(df['Target'].value_counts())
print(f"\nclass ratio: {df['Target'].value_counts()[1] / df['Target'].value_counts()[0]:.2f}")
if df['Target'].value_counts()[1] / df['Target'].value_counts()[0] < 0.5 or df['target'].value_counts()[1] / df['target'].value_counts()[0] > 2:
    print("Target is IMBALANCED - Use F1 Score")
else:
    print("Target is BALANCED - Can use Accuracy")

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns=['Target'])
y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, accuracy_score

stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
fold_models = []

for fold, (train_idx, val_idx) in enumerate(stratified_kfold.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, random_state=42, verbose=0)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    score = f1_score(y_val, y_pred)
    scores.append(score)
    fold_models.append(model)

    print(f"Fold {fold}: F1 Score = {score:.4f}")

print(f"\nAverage F1 Score: {np.mean(scores):.4f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = np.mean([model.feature_importances_ for model in fold_models], axis=0)
importance_df = pd.DataFrame({'feature': X.columns, 'importance': feature_importance}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'], importance_df['importance'], color='gold')
plt.xlabel('Feature Importance')
plt.title('CatBoost Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
golden_feature = importance_df.iloc[-1]['feature']
print(f"The Golden Feature is: {golden_feature}")
print(f"Importance score: {importance_df.iloc[-1]['importance']:.4f}")

In [ ]:
# Task Bonus: Write your code here:
X_golden = df[[golden_feature]]
y_golden = df['target']

stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
golden_scores = []

for fold, (train_idx, val_idx) in enumerate(stratified_kfold.split(X_golden, y_golden), 1):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y_golden.iloc[train_idx], y_golden.iloc[val_idx]

    model = CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, random_state=42, verbose=0)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    score = f1_score(y_val, y_pred)
    golden_scores.append(score)

    print(f"Fold {fold}: F1 Score = {score:.4f}")

print(f"\nAverage F1 Score (Golden Feature Only): {np.mean(golden_scores):.4f}")
print(f"\nComparison:")
print(f"  Full Model F1 Score:           {np.mean(scores):.4f}")
print(f"  Golden Feature Only F1 Score:  {np.mean(golden_scores):.4f}")